# PyTorch 入門 — インタラクティブノートブック

テンソル操作・自動微分・MLP の実装・訓練ループを学びます。  
依存: `pip install torch matplotlib scikit-learn`

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__}")
print(f"使用デバイス: {device}")

## 1. テンソル操作

In [ ]:
# テンソルの作成
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.randn(2, 2)

print("a:")
print(a)
print(f"  形状: {a.shape}, dtype: {a.dtype}")

print("\n行列積 a @ b:")
print(a @ b)

print(f"\n転置:\n{a.T}")
print(f"合計: {a.sum():.4f}, 平均: {a.mean():.4f}")

In [ ]:
# 自動微分
x = torch.tensor(3.0, requires_grad=True)
y = x**3 - 2*x**2 + x   # f(x) = x^3 - 2x^2 + x

y.backward()             # dy/dx を計算

print(f"x = {x.item():.1f}")
print(f"f(x) = {y.item():.4f}")
print(f"df/dx (autograd) = {x.grad.item():.4f}")
print(f"df/dx (解析解)   = {3*x.item()**2 - 4*x.item() + 1:.4f}")

## 2. MLP の定義

In [ ]:
class IrisMLP(nn.Module):
    """Iris 分類用 MLP。"""

    def __init__(self, input_dim: int = 4, hidden: int = 32, n_classes: int = 3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

model = IrisMLP().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\n総パラメータ数: {total_params:,}")

# ダミー入力でフォワードパスを確認
dummy = torch.randn(8, 4).to(device)
out = model(dummy)
print(f"出力形状: {out.shape}")

## 3. データ準備と訓練ループ

In [ ]:
class IrisDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


iris = load_iris()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(iris.data)

dataset = IrisDataset(X_scaled, iris.target)
n_train = int(0.8 * len(dataset))
train_ds, val_ds = random_split(dataset, [n_train, len(dataset) - n_train],
                                generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=16)

print(f"訓練: {len(train_ds)} サンプル, 検証: {len(val_ds)} サンプル")

In [ ]:
model = IrisMLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

train_losses, val_accs = [], []
N_EPOCHS = 100

for epoch in range(N_EPOCHS):
    # --- 訓練 ---
    model.train()
    epoch_loss = 0.0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()        # 勾配をリセット
        loss = criterion(model(X_b), y_b)
        loss.backward()              # 誤差逆伝播
        optimizer.step()             # パラメータ更新
        epoch_loss += loss.item() * len(X_b)
    train_losses.append(epoch_loss / len(train_ds))

    # --- 検証 ---
    model.eval()
    correct = 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            correct += (model(X_b).argmax(1) == y_b).sum().item()
    val_accs.append(correct / len(val_ds))
    scheduler.step()

print(f"最終 訓練損失: {train_losses[-1]:.4f}")
print(f"最終 検証精度: {val_accs[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, label="訓練損失", color="steelblue")
axes[0].set_xlabel("エポック"); axes[0].set_ylabel("損失")
axes[0].set_title("訓練損失の推移"); axes[0].legend()

axes[1].plot(val_accs, label="検証精度", color="coral")
axes[1].axhline(y=max(val_accs), color="gray", linestyle="--",
                label=f"最大精度: {max(val_accs):.4f}")
axes[1].set_xlabel("エポック"); axes[1].set_ylabel("精度")
axes[1].set_title("検証精度の推移"); axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 詳細レポート
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in val_loader:
        X_b = X_b.to(device)
        all_preds.extend(model(X_b).argmax(1).cpu().numpy())
        all_labels.extend(y_b.numpy())

print(classification_report(all_labels, all_preds,
                             target_names=iris.target_names))

## 4. モデルの保存と読み込み

In [ ]:
# 保存
torch.save(model.state_dict(), "iris_mlp.pth")
print("保存完了: iris_mlp.pth")

# 読み込み
model_loaded = IrisMLP().to(device)
model_loaded.load_state_dict(torch.load("iris_mlp.pth", map_location=device))
model_loaded.eval()

# 予測が一致することを確認
sample = torch.randn(5, 4).to(device)
with torch.no_grad():
    out_orig = model(sample).argmax(1)
    out_load = model_loaded(sample).argmax(1)

print("元のモデル:", out_orig.cpu().numpy())
print("読み込みモデル:", out_load.cpu().numpy())
print("一致:", torch.equal(out_orig, out_load))

## 5. 活性化関数の比較

In [ ]:
x = torch.linspace(-4, 4, 200)

activations = {
    "ReLU":    F.relu(x),
    "Sigmoid": torch.sigmoid(x),
    "Tanh":    torch.tanh(x),
    "GELU":    F.gelu(x),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
colors = ["steelblue", "coral", "seagreen", "mediumpurple"]

for ax, (name, y), color in zip(axes.flat, activations.items(), colors):
    ax.plot(x.numpy(), y.numpy(), color=color, lw=2)
    ax.axhline(0, color="k", lw=0.5)
    ax.axvline(0, color="k", lw=0.5)
    ax.set_title(name)
    ax.set_xlabel("x")
    ax.set_ylabel("f(x)")
    ax.grid(True, alpha=0.3)

plt.suptitle("活性化関数の比較", fontsize=14)
plt.tight_layout()
plt.show()